# SanteScope — 04_jumelles.ipynb

Compute twin commune matching via KNN (sklearn NearestNeighbors) on normalized health indicators.

**Phase 02, Plan 02, Task 1** — Region-priority twin selection with improvement signal detection.

---

In [1]:
import pandas as pd
import numpy as np
import json
import time
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

# Path-agnostic detection
if Path('notebooks/data/processed').exists():
    PROCESSED = Path('notebooks/data/processed')
elif Path('data/processed').exists():
    PROCESSED = Path('data/processed')
else:
    raise FileNotFoundError("Cannot find data/processed directory")

df = pd.read_parquet(PROCESSED / 'communes_master.parquet')
print(f"Loaded: {df.shape[0]} communes, {df.shape[1]} columns")
assert 'region' in df.columns, "region column missing — run 03_domino first"
assert 'densite_hab_km2' in df.columns, "density column missing — run 03_domino first"
print(f"Region coverage: {df['region'].notna().sum()} / {len(df)}")
print(f"Density coverage: {df['densite_hab_km2'].notna().sum()} / {len(df)}")


Loaded: 34969 communes, 46 columns
Region coverage: 34877 / 34969
Density coverage: 34876 / 34969


## Section 2 — Feature Matrix Preparation

Features jumelles: APL, pauvrete, pct_75_plus, log(population), densite.
Distance euclidienne sur valeurs normalisees min-max.

Choix: on utilise les indicateurs bruts normalises (pas le score de vulnerabilite)
car le score est trop comprime (1.0-6.1) a cause de l'imputation pauvrete a 87.8%.
Les indicateurs bruts preservent la variance entre communes.

In [2]:
TWIN_FEATURES = ['apl', 'taux_pauvrete', 'pct_75_plus', 'log_pop', 'densite_hab_km2']

df['log_pop'] = np.log1p(df['population'])

features = df[TWIN_FEATURES].copy()
for col in TWIN_FEATURES:
    features[col] = features[col].fillna(features[col].median())

scaler = MinMaxScaler()
X = scaler.fit_transform(features)

print(f"Feature matrix: {X.shape}")
print(f"Feature ranges after scaling:")
for i, col in enumerate(TWIN_FEATURES):
    print(f"  {col}: [{X[:, i].min():.3f}, {X[:, i].max():.3f}]")


Feature matrix: (34969, 5)
Feature ranges after scaling:
  apl: [0.000, 1.000]
  taux_pauvrete: [0.000, 1.000]
  pct_75_plus: [0.000, 1.000]
  log_pop: [0.000, 1.000]
  densite_hab_km2: [0.000, 1.000]


## Section 3 — KNN Fit

NearestNeighbors avec ball_tree (O(n log n)) sur la matrice 35K x 5.
n_neighbors=51 pour avoir 50 voisins + self.

In [3]:
t0 = time.time()
nbrs = NearestNeighbors(n_neighbors=51, algorithm='ball_tree', metric='euclidean')
nbrs.fit(X)
distances, indices = nbrs.kneighbors(X)
elapsed = time.time() - t0
print(f"KNN fit + query: {elapsed:.2f}s for {len(df)} communes")
print(f"Distance stats (excluding self): min={distances[:, 1].min():.4f}, median={np.median(distances[:, 1]):.4f}, max={distances[:, 1].max():.4f}")


KNN fit + query: 2.83s for 34969 communes
Distance stats (excluding self): min=0.0000, median=0.0043, max=0.3835


## Section 4 — Improvement Signal Detection

Signal d'amelioration: MSP installee OU evolution APL > 0.3.
Ces signaux sont detectables dans nos donnees et representent des actions concretes.

In [4]:
has_improved = (df['has_msp'] == True) | (df['apl_evolution'] > 0.3)
print(f"{has_improved.sum()} communes avec signal d'amelioration")
print(f"  - MSP: {(df['has_msp'] == True).sum()}")
print(f"  - APL > 0.3: {(df['apl_evolution'] > 0.3).sum()}")
print(f"  - Les deux: {((df['has_msp'] == True) & (df['apl_evolution'] > 0.3)).sum()}")

# Build region index for same-region priority
region_indices = {}
for reg in df['region'].dropna().unique():
    region_indices[reg] = set(df.index[df['region'] == reg].tolist())
print(f"\n{len(region_indices)} regions indexees")
for reg, idx_set in sorted(region_indices.items(), key=lambda x: -len(x[1]))[:5]:
    print(f"  Region {reg}: {len(idx_set)} communes")


5263 communes avec signal d'amelioration
  - MSP: 2358
  - APL > 0.3: 3129
  - Les deux: 224

19 regions indexees
  Region 44: 5115 communes
  Region 76: 4446 communes
  Region 75: 4293 communes
  Region 84: 4025 communes
  Region 32: 3782 communes


## Section 5 — Twin Selection with Region Priority

Priorite regionale: chercher d'abord dans la meme region, puis national si <5 jumelles ameliorees.
Si <3 jumelles au total, ajouter les voisins les plus proches (meme sans amelioration).

Similarite = 1 - (distance / max_distance_possible).
Max distance en espace normalise 5D = sqrt(5) ~ 2.236.

In [5]:
MAX_TWINS = 5
SIMILARITY_SCALE = np.sqrt(len(TWIN_FEATURES))

def build_twin(j, dist):
    twin_row = df.iloc[j]
    sim = round(max(1 - (dist / SIMILARITY_SCALE), 0), 2)
    actions = []
    if twin_row['has_msp']:
        actions.append(f"MSP installee ({int(twin_row['nb_msp'])} MSP)")
    if pd.notna(twin_row['apl_evolution']) and twin_row['apl_evolution'] > 0.3:
        apl_22 = f"{twin_row['apl_2022']:.1f}" if pd.notna(twin_row['apl_2022']) else "?"
        apl_23 = f"{twin_row['apl_2023']:.1f}" if pd.notna(twin_row['apl_2023']) else "?"
        actions.append(f"APL: {apl_22} -> {apl_23}")
    return {
        'code': str(twin_row['code_commune']),
        'nom': str(twin_row['nom_commune']),
        'similarite': sim,
        'actions': actions,
        'has_msp': bool(twin_row['has_msp']),
        'apl_avant': round(float(twin_row['apl_2022']), 1) if pd.notna(twin_row['apl_2022']) else None,
        'apl_apres': round(float(twin_row['apl_2023']), 1) if pd.notna(twin_row['apl_2023']) else None,
    }

t0 = time.time()
jumelles_list = []

for i in range(len(df)):
    if i % 5000 == 0 and i > 0:
        elapsed = time.time() - t0
        print(f"  {i}/{len(df)} communes traitees ({elapsed:.1f}s)")
    
    commune_region = df.iloc[i]['region']
    neighbors = indices[i][1:]  # skip self
    neighbor_dists = distances[i][1:]
    
    twins = []
    added_indices = set()
    
    # Step 1: same-region improved twins
    if commune_region and commune_region in region_indices:
        reg_set = region_indices[commune_region]
        for j, dist in zip(neighbors, neighbor_dists):
            if len(twins) >= MAX_TWINS:
                break
            if j in reg_set and has_improved.iloc[j]:
                twins.append(build_twin(j, dist))
                added_indices.add(j)
    
    # Step 2: expand nationally if <5 improved twins
    if len(twins) < MAX_TWINS:
        for j, dist in zip(neighbors, neighbor_dists):
            if len(twins) >= MAX_TWINS:
                break
            if j not in added_indices and has_improved.iloc[j]:
                twins.append(build_twin(j, dist))
                added_indices.add(j)
    
    # Step 3: if <3 twins, add best-match non-improved neighbors
    if len(twins) < 3:
        for j, dist in zip(neighbors, neighbor_dists):
            if len(twins) >= 3:
                break
            if j not in added_indices:
                twins.append(build_twin(j, dist))
                added_indices.add(j)
    
    jumelles_list.append(twins)

elapsed = time.time() - t0
print(f"Twin selection complete: {elapsed:.1f}s for {len(df)} communes")
df['jumelles'] = jumelles_list


  5000/34969 communes traitees (1.7s)


  10000/34969 communes traitees (3.4s)


  15000/34969 communes traitees (5.1s)


  20000/34969 communes traitees (6.9s)


  25000/34969 communes traitees (8.6s)


  30000/34969 communes traitees (10.3s)


Twin selection complete: 12.0s for 34969 communes


## Section 6 — Validation

In [6]:
assert df['jumelles'].apply(lambda x: isinstance(x, list)).all(), "jumelles not all lists"
assert df['jumelles'].apply(lambda x: len(x) <= 5).all(), "Twin count exceeds 5"
assert df['jumelles'].apply(lambda x: all('similarite' in t for t in x)).all(), "Missing similarite key"
assert df['jumelles'].apply(lambda x: all('actions' in t for t in x)).all(), "Missing actions key"

twin_counts = df['jumelles'].apply(len)
print("Twin count distribution:")
print(twin_counts.describe())
print(f"\nCommunes with 0 twins: {(twin_counts == 0).sum()}")
print(f"Communes with 1-2 twins: {twin_counts.between(1, 2).sum()}")
print(f"Communes with 3-5 twins: {twin_counts.between(3, 5).sum()}")

has_improved_twin = df['jumelles'].apply(lambda x: any(t['actions'] for t in x))
print(f"\nCommunes with at least 1 improved twin: {has_improved_twin.sum()}")

# Check similarity range
all_sims = [t['similarite'] for twins in df['jumelles'] for t in twins]
print(f"\nSimilarity range: [{min(all_sims):.2f}, {max(all_sims):.2f}]")
print(f"Mean similarity: {np.mean(all_sims):.2f}")


Twin count distribution:
count    34969.000000
mean         4.226229
std          0.926943
min          3.000000
25%          3.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: jumelles, dtype: float64

Communes with 0 twins: 0
Communes with 1-2 twins: 0
Communes with 3-5 twins: 34969

Communes with at least 1 improved twin: 33274

Similarity range: [0.65, 1.00]
Mean similarity: 0.99


## Section 7 — Save

In [7]:
# Convert numpy arrays to native Python lists for clean parquet serialization
def ensure_native(twins):
    result = []
    for t in twins:
        d = {}
        for k, v in t.items():
            if hasattr(v, 'tolist'):
                d[k] = v.tolist()
            else:
                d[k] = v
        result.append(d)
    return result

df['jumelles'] = df['jumelles'].apply(ensure_native)

df.to_parquet(PROCESSED / 'communes_master.parquet', index=False)
print(f"Saved: {df.shape[0]} communes, {df.shape[1]} columns")
print(f"New column: jumelles (list of dicts, up to 5 per commune)")


Saved: 34969 communes, 46 columns
New column: jumelles (list of dicts, up to 5 per commune)
